# 5. Metadata-Filtered RAG
**Industry:** Law Firms

Filter retrieval results using metadata (e.g., search precedents only from a specific jurisdiction).

In [1]:
!pip install langchain langchain-openai chromadb sentence-transformers langchain-community


[notice] A new release of pip is available: 24.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
import os
import dotenv
dotenv.load_dotenv(r"D:/Internship/Teach-ai/Backend/.env")
from langchain_openai import AzureChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# Mock Case Law documents with metadata
documents = [
    Document(page_content="Case A: The court ruled that software is not inherently patentable.", metadata={"jurisdiction": "bombay_hc", "year": 2023}),
    Document(page_content="Case B: Software can be patented if it shows a technical advancement.", metadata={"jurisdiction": "delhi_hc", "year": 2023}),
    Document(page_content="Case C: Previous ruling reversed. Hardware is required for patent.", metadata={"jurisdiction": "bombay_hc", "year": 2019})
]

vectorstore = Chroma.from_documents(documents, embedding=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2"))

# Retriever without filtering
unfiltered_retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
print("Unfiltered Results:")
for doc in unfiltered_retriever.invoke("software patentability"): 
    print(f"- {doc.page_content} ({doc.metadata})")

print("\n---\n")

# Retriever WITH metadata filtering
filtered_retriever = vectorstore.as_retriever(search_kwargs={"k": 2, "filter": {"jurisdiction": "bombay_hc", "year": 2023}})
print("Filtered Results (Bombay HC, 2023):")
for doc in filtered_retriever.invoke("software patentability"): 
    print(f"- {doc.page_content} ({doc.metadata})")